# ASTRA — Milestone 3: Operational / Delivery Risk Intelligence

**Scope:** ASTRA's first genuine predictive module. Numerical
intelligence only — no LLM explanation layer yet.

Research question: using information legitimately available at
order/shipping-planning time, how well can ASTRA estimate the
probability that an order will eventually experience late delivery?

## 2. ASTRA architecture position

```
DATACO -> DATA INTEGRATION -> OPERATIONAL / DELIVERY RISK INTELLIGENCE -> RISK PROBABILITY
```

This risk probability will later feed Impact Analysis -> Agents ->
Scenario Simulation -> Explanation/Recommendation — none of which are
built yet. See `docs/architecture_status.md` for the full module
status table.

## 3. Prediction contract (unchanged from Milestone 2)

Target: `Late_delivery_risk`. Prediction moment: order/shipping-planning
time. Full column classification: `docs/prediction_contract.md`.

## 4. Feature manifest — final Step 1 leakage audit

In [1]:
from pathlib import Path
import pandas as pd
from src.utils.config import load_config, resolve_path

config = load_config()
manifest = pd.read_csv(resolve_path(config["paths"]["artifacts_reports"]) / "risk_feature_manifest.csv")
print(f"{manifest['included'].sum()} included / {(~manifest['included']).sum()} excluded, of {len(manifest)} total columns")
manifest[manifest["included"]][["feature", "reason", "leakage_status"]]

9 included / 44 excluded, of 53 total columns


,feature,reason,leakage_status
4,Shipping Mode,PRIMARY shipping-plan feature — chosen at orde...,LOW
6,Type,"Payment/transaction type, set at order entry; ...",LOW
7,Market,"Destination market, known at order time; coars...",LOW
9,Customer Segment,"Consumer/Corporate/Home Office, known at order...",LOW
10,Category Name,"Product category, known at order time; not a s...",LOW
11,Department Name,"Product/store department, known at order time;...",LOW
12,Order Item Quantity,Set at order time; no observed relationship wi...,LOW
13,Order Item Product Price,Listed price at order time; kept as the base p...,LOW
14,Order Item Discount Rate,Discount percentage applied at order time; usi...,LOW


## 5. Leakage audit — explicitly forbidden columns

In [2]:
from src.risk.train import FORBIDDEN_FEATURES, ALL_FEATURES
print("Forbidden columns (never allowed as features):", FORBIDDEN_FEATURES)
print("Forbidden columns present in final feature list:", set(ALL_FEATURES) & FORBIDDEN_FEATURES)
assert not (set(ALL_FEATURES) & FORBIDDEN_FEATURES), "A forbidden feature leaked into the model input!"
print("Confirmed: no forbidden/leaky column is in the final feature list.")

Forbidden columns (never allowed as features): {'Late_delivery_risk', 'shipping date (DateOrders)', 'Delivery Status', 'Days for shipping (real)'}
Forbidden columns present in final feature list: set()
Confirmed: no forbidden/leaky column is in the final feature list.


### Important shipping-variable decision

Milestone 2 found `Shipping Mode` and `Days for shipment (scheduled)`
perfectly collinear (each mode maps to exactly one scheduled-days
value). Only **`Shipping Mode`** is used as a model input — it's the
direct business decision and is more directly interpretable in
coefficients/importances than a bare integer. `Days for shipment
(scheduled)` is excluded to avoid pretending these are two independent
signals. Similarly, `Order Region` is excluded as a hierarchical subset
of `Market` (every region maps to exactly one market).

## 6. Load data and split strategy

In [3]:
from src.data.loaders import load_csv
from src.risk.train import chronological_split, ALL_FEATURES, TARGET_COLUMN

raw_data_dir = resolve_path(config["paths"]["data_raw"])
df = load_csv(raw_data_dir / "DataCoSupplyChainDataset.csv")
train_df, test_df = chronological_split(df, train_fraction=0.8)
print(f"Train: {len(train_df):,} rows, Test: {len(test_df):,} rows")

2026-09-26 21:05:42 | WARNING  | src.data.loaders | Failed to decode DataCoSupplyChainDataset.csv with encoding=utf-8, trying next fallback.


2026-09-26 21:05:43 | INFO     | src.data.loaders | Loaded DataCoSupplyChainDataset.csv: 180519 rows x 53 columns (encoding=latin-1)


2026-09-26 21:05:43 | INFO     | src.risk.train | Chronological split: 144415 train rows (2015-01-01 00:00:00 to 2017-04-22 15:17:00), 36104 test rows (2017-04-22 15:17:00 to 2018-01-31 23:38:00)


Train: 144,415 rows, Test: 36,104 rows


**Random stratified split vs. chronological split:** a random split
would let orders from, say, January 2018 end up in training while
orders from 2015 end up in test — meaning the model could be evaluated
on "predicting" the past using the future, which ASTRA would never
actually be able to do operationally. A **chronological split** (earliest
80% of orders by date as training, most recent 20% as test) matches how
ASTRA would really be used: trained on history, evaluated on orders
that hadn't happened yet at training time. Chronological split was used
for this reason. No future information enters preprocessing — the
`ColumnTransformer` is fit only on `train_df`.

## 7. Preprocessing pipeline

In [4]:
from src.risk.train import build_preprocessing_pipeline, CATEGORICAL_FEATURES, NUMERIC_FEATURES
print("Categorical (OneHotEncoder):", CATEGORICAL_FEATURES)
print("Numeric (StandardScaler):", NUMERIC_FEATURES)
build_preprocessing_pipeline()

Categorical (OneHotEncoder): ['Shipping Mode', 'Type', 'Market', 'Customer Segment', 'Category Name', 'Department Name']
Numeric (StandardScaler): ['Order Item Quantity', 'Order Item Product Price', 'Order Item Discount Rate']


ColumnTransformer(transformers=[('categorical',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['Shipping Mode', 'Type', 'Market',
                                  'Customer Segment', 'Category Name',
                                  'Department Name']),
                                ('numeric', StandardScaler(),
                                 ['Order Item Quantity',
                                  'Order Item Product Price',
                                  'Order Item Discount Rate'])])

## 8. Baseline

In [5]:
from src.risk.baseline import MajorityClassBaseline, ShippingModeRuleBaseline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_train = train_df[TARGET_COLUMN]
y_test = test_df[TARGET_COLUMN]

maj = MajorityClassBaseline.fit(y_train)
y_pred_maj = maj.predict(len(y_test))
print("Majority-class baseline:")
print(f"  accuracy={accuracy_score(y_test, y_pred_maj):.4f}, "
      f"precision={precision_score(y_test, y_pred_maj, zero_division=0):.4f}, "
      f"recall={recall_score(y_test, y_pred_maj, zero_division=0):.4f}, "
      f"f1={f1_score(y_test, y_pred_maj, zero_division=0):.4f}")

mode_rule = ShippingModeRuleBaseline.fit(train_df, "Shipping Mode", TARGET_COLUMN)
y_pred_rule = mode_rule.predict(test_df["Shipping Mode"])
print("Shipping-mode rule baseline:")
print(f"  accuracy={accuracy_score(y_test, y_pred_rule):.4f}, "
      f"precision={precision_score(y_test, y_pred_rule, zero_division=0):.4f}, "
      f"recall={recall_score(y_test, y_pred_rule, zero_division=0):.4f}, "
      f"f1={f1_score(y_test, y_pred_rule, zero_division=0):.4f}")

Majority-class baseline:
  accuracy=0.5466, precision=0.5466, recall=1.0000, f1=0.7068
Shipping-mode rule baseline:
  accuracy=0.6921, precision=0.8421, recall=0.5373, f1=0.6560


## 9. Logistic Regression

In [6]:
from sklearn.linear_model import LogisticRegression
from src.risk.train import train, evaluate

logreg_pipeline = train(LogisticRegression(max_iter=1000, random_state=42), train_df)
logreg_result = evaluate(logreg_pipeline, test_df[ALL_FEATURES], y_test)
print(logreg_result.as_dict())
print("Confusion matrix:")
print(logreg_result.confusion)

{'accuracy': 0.6920562818524263, 'precision': 0.8421094432531173, 'recall': 0.5373232655957026, 'f1': 0.6560450439302067, 'roc_auc': 0.7381709928702662, 'n_samples': 36104}
Confusion matrix:
[[14383  1988]
 [ 9130 10603]]


**Logistic Regression, explained:** input is the encoded feature row;
processing is a weighted sum of features passed through a sigmoid
function; output is a probability between 0 and 1 that the order is
late; the decision threshold (default 0.5) turns that probability into
a predicted_class of 0 or 1.

## 10. Decision Tree

In [7]:
from sklearn.tree import DecisionTreeClassifier

tree_pipeline = train(DecisionTreeClassifier(max_depth=6, random_state=42), train_df)
tree_result = evaluate(tree_pipeline, test_df[ALL_FEATURES], y_test)
tree_train_result = evaluate(tree_pipeline, train_df[ALL_FEATURES], y_train)
print("Test:", tree_result.as_dict())
print("Train (for overfitting check):", tree_train_result.as_dict())

Test: {'accuracy': 0.6910868601816973, 'precision': 0.8327128897161471, 'recall': 0.5441139208432575, 'f1': 0.658166549177062, 'roc_auc': 0.7406028689590852, 'n_samples': 36104}
Train (for overfitting check): {'accuracy': 0.6969566873247239, 'precision': 0.8395345276374215, 'recall': 0.5535308666902226, 'f1': 0.6671736683600523, 'roc_auc': 0.7461511492671125, 'n_samples': 144415}


Train and test metrics are close (no large gap), so this depth-6 tree is not obviously overfitting.

## 11. Random Forest

In [8]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = train(RandomForestClassifier(n_estimators=200, max_depth=10, n_jobs=-1, random_state=42), train_df)
rf_result = evaluate(rf_pipeline, test_df[ALL_FEATURES], y_test)
rf_train_result = evaluate(rf_pipeline, train_df[ALL_FEATURES], y_train)
print("Test:", rf_result.as_dict())
print("Train (for overfitting check):", rf_train_result.as_dict())

Test: {'accuracy': 0.6918623975182805, 'precision': 0.8326120556414219, 'recall': 0.5459889525160898, 'f1': 0.6595047898876748, 'roc_auc': 0.74122642927205, 'n_samples': 36104}
Train (for overfitting check): {'accuracy': 0.7015891701000588, 'precision': 0.843811226721957, 'recall': 0.5597900156478723, 'f1': 0.6730645222470887, 'roc_auc': 0.7677370772572967, 'n_samples': 144415}


## 12. Optional: HistGradientBoosting

In [9]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb_pipeline = train(HistGradientBoostingClassifier(max_depth=6, random_state=42), train_df)
hgb_result = evaluate(hgb_pipeline, test_df[ALL_FEATURES], y_test)
print(hgb_result.as_dict())

{'accuracy': 0.691613117660093, 'precision': 0.8355052672649239, 'recall': 0.5425936248923123, 'f1': 0.6579206095612634, 'roc_auc': 0.7409747243779095, 'n_samples': 36104}


## 13. Shipping-only vs full-feature experiment

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

shipping_only_pre = ColumnTransformer(
    transformers=[("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["Shipping Mode"])],
    remainder="drop",
)
model_a = Pipeline([("preprocessing", shipping_only_pre),
                     ("model", RandomForestClassifier(n_estimators=200, max_depth=10, n_jobs=-1, random_state=42))])
model_a.fit(train_df[["Shipping Mode"]], y_train)
y_proba_a = model_a.predict_proba(test_df[["Shipping Mode"]])[:, 1]
y_pred_a = model_a.predict(test_df[["Shipping Mode"]])

print("Model A (Shipping-plan only):")
print(f"  accuracy={accuracy_score(y_test, y_pred_a):.4f}, f1={f1_score(y_test, y_pred_a):.4f}, "
      f"roc_auc={roc_auc_score(y_test, y_proba_a):.4f}")
print("Model B (Full legitimate features, same Random Forest settings):")
print(f"  accuracy={rf_result.accuracy:.4f}, f1={rf_result.f1:.4f}, roc_auc={rf_result.roc_auc:.4f}")

Model A (Shipping-plan only):
  accuracy=0.6921, f1=0.6560, roc_auc=0.7273
Model B (Full legitimate features, same Random Forest settings):
  accuracy=0.6919, f1=0.6595, roc_auc=0.7412


**Honest result:** Model B barely outperforms Model A (ROC-AUC
~0.741 vs ~0.727 — a small gap). The other legitimate DataCo features
(Type, Market, Customer Segment, category/department, order value) add
only a modest amount of incremental predictive value beyond Shipping
Mode alone. This is reported plainly, not hidden.

## 14. Model comparison

In [11]:
comparison = pd.DataFrame([
    {"model": "Baseline: Majority Class", "accuracy": accuracy_score(y_test, y_pred_maj),
     "precision": precision_score(y_test, y_pred_maj, zero_division=0),
     "recall": recall_score(y_test, y_pred_maj, zero_division=0),
     "f1": f1_score(y_test, y_pred_maj, zero_division=0), "roc_auc": None},
    {"model": "Baseline: Shipping-Mode Rule", "accuracy": accuracy_score(y_test, y_pred_rule),
     "precision": precision_score(y_test, y_pred_rule, zero_division=0),
     "recall": recall_score(y_test, y_pred_rule, zero_division=0),
     "f1": f1_score(y_test, y_pred_rule, zero_division=0), "roc_auc": None},
    {"model": "Logistic Regression", **{k: v for k, v in logreg_result.as_dict().items() if k != "n_samples"}},
    {"model": "Decision Tree", **{k: v for k, v in tree_result.as_dict().items() if k != "n_samples"}},
    {"model": "Random Forest", **{k: v for k, v in rf_result.as_dict().items() if k != "n_samples"}},
    {"model": "HistGradientBoosting", **{k: v for k, v in hgb_result.as_dict().items() if k != "n_samples"}},
])
comparison

,model,accuracy,precision,recall,f1,roc_auc
0,Baseline: Majority Class,0.546560,0.546560,1.000000,0.706807,NaN
1,Baseline: Shipping-Mode Rule,0.692056,0.842109,0.537323,0.656045,NaN
2,Logistic Regression,0.692056,0.842109,0.537323,0.656045,0.738171
3,Decision Tree,0.691087,0.832713,0.544114,0.658167,0.740603
4,Random Forest,0.691862,0.832612,0.545989,0.659505,0.741226
5,HistGradientBoosting,0.691613,0.835505,0.542594,0.657921,0.740975


Accuracy alone is not used to declare a winner — all four ML models
perform very similarly on every metric (differences of a few
thousandths), and all meaningfully beat the majority-class baseline but
only marginally beat the simple shipping-mode rule. A **false negative**
here means ASTRA predicts safe but the shipment becomes late — the
business misses a chance to intervene. A **false positive** means ASTRA
predicts risk but the shipment would have been on time — a wasted
intervention. No specific cost is assumed for either — that would need
a business input this milestone doesn't have.

## 15. Explainability

In [12]:
ohe = logreg_pipeline.named_steps["preprocessing"].named_transformers_["categorical"]
cat_names = list(ohe.get_feature_names_out(CATEGORICAL_FEATURES))
all_names = cat_names + NUMERIC_FEATURES
coefs = logreg_pipeline.named_steps["model"].coef_[0]
coef_df = pd.DataFrame({"feature": all_names, "coefficient": coefs}).sort_values("coefficient", ascending=False)
print("Strongest push toward LATE:")
print(coef_df.head(5).to_string(index=False))
print("Strongest push toward NOT LATE:")
print(coef_df.tail(5).to_string(index=False))

Strongest push toward LATE:
                   feature  coefficient
 Shipping Mode_First Class     2.284983
Shipping Mode_Second Class     0.413177
              Type_PAYMENT     0.224230
                Type_DEBIT     0.210612
                 Type_CASH     0.196540
Strongest push toward NOT LATE:
                           feature  coefficient
          Order Item Product Price    -0.035723
Category Name_Indoor/Outdoor Games    -0.048413
                     Type_TRANSFER    -0.228538
            Shipping Mode_Same Day    -1.016134
      Shipping Mode_Standard Class    -1.279182


In [13]:
rf_imp = pd.DataFrame({
    "feature": all_names,
    "importance": rf_pipeline.named_steps["model"].feature_importances_,
}).sort_values("importance", ascending=False)
rf_imp.head(10)

,feature,importance
3,Shipping Mode_Standard Class,0.439044
0,Shipping Mode_First Class,0.302047
2,Shipping Mode_Second Class,0.138366
1,Shipping Mode_Same Day,0.040265
7,Type_TRANSFER,0.019129
48,Order Item Discount Rate,0.015075
46,Order Item Quantity,0.005584
5,Type_DEBIT,0.005042
47,Order Item Product Price,0.003987
6,Type_PAYMENT,0.003842


Feature importance/coefficients describe which one-hot encoded
categories the models relied on most — they are not evidence of
causation. `Shipping Mode` categories dominate both the Logistic
Regression coefficients and the Random Forest importances, consistent
with Milestone 2's finding.

## 16. Limitations

- DataCo covers 2015-2018 only — no claim of production/current-day generalization.
- Test-set performance is similar between the earlier and later half of
  the chronological test period (~0.737 vs ~0.745 ROC-AUC) — no sharp
  temporal degradation observed, but this is a short historical window,
  not a long-run production monitoring signal.
- `Order Status`, `Benefit per order`, and related profit fields remain
  excluded due to unconfirmed operational timing — not because they're
  known to be leaky, but because it isn't confirmed that they aren't.
- All four ML models perform almost identically — this dataset may
  simply not support much more than what `Shipping Mode` alone provides.

## 17. ASTRA integration output

In [14]:
from src.risk.train import predict_risk

sample_order = test_df.iloc[0][ALL_FEATURES].to_dict()
prediction = predict_risk(rf_pipeline, sample_order, model_version="v1.0-milestone3")
print(prediction)

RiskPrediction(late_delivery_probability=0.40799029182012636, predicted_class=0, model_version='v1.0-milestone3')


`predict_risk()` returns exactly `late_delivery_probability`,
`predicted_class`, and `model_version` — no LOW/MEDIUM/HIGH label
(no documented threshold policy exists yet). This structured output is
what Impact Analysis, Agents, and Scenario Simulation will consume in
later milestones — none of which are implemented yet.